# 04. Baseball Savant 영상 다운로드 (코랩용)
- 전체 1,185,600개를 5대(코랩 2 + 로컬 PC 3)로 분할 다운로드
- **코랩 A = 슬롯 0** (0 ~ 237,120번)
- **코랩 B = 슬롯 1** (237,120 ~ 474,240번)
- 슬롯 변경하려면 cell-8의 `MY_SLOT` 만 수정
- 배치 200개 → zip → MyDrive 저장 / 체크포인트로 이어서 실행 가능

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os

# ══════════════════════════════════════════════════════
# ★ 여기만 수정 ★
# 나 (프로플러스): MY_SEASONS = [2021, 2022, 2023] / BATCH_SIZE = 200
# 팀원 (무료)   : MY_SEASONS = [2024, 2025]        / BATCH_SIZE = 100
MY_SEASONS   = [2021, 2022, 2023]
X_MAX_BATTER = 9
BATCH_SIZE   = 200
# ══════════════════════════════════════════════════════

# parquet 파일 ID (2021~2025 순서)
PARQUET_FILE_IDS = {
    2021: '1KT17izBcO92VXTIr0LyyWGl-0bKHW7sS',
    2022: '16ak1N09M0ptyq4xjR-mpCBnY-L1dQadE',
    2023: '1bX2n3hpptTBZszPyB5EY2Ov0lCI4nM8I',

    2024: '17-ZhSO18-vjkSFXOZtQKEVTYnPmGjMuY',
    2025: '1uW_apJ4IC63FvTvK6BkZYBaAKA8ge5_L',
}

# zip/progress/play_id는 내 MyDrive에 저장
MY_BASE      = '/content/drive/MyDrive/MLB_pitcher'
VIDEO_DIR    = os.path.join(MY_BASE, 'video_zips')
PROGRESS_DIR = os.path.join(MY_BASE, 'data')
PARQUET_DIR  = '/content/parquets'   # 코랩 임시 저장
TEMP_DIR     = '/content/tmp_videos'

SEASON_KEY   = '_'.join(map(str, MY_SEASONS))
PROGRESS_CSV = os.path.join(PROGRESS_DIR, f'progress_{SEASON_KEY}.csv')
PLAY_ID_CSV  = os.path.join(PROGRESS_DIR, f'play_ids_{SEASON_KEY}.csv')

os.makedirs(VIDEO_DIR,    exist_ok=True)
os.makedirs(PROGRESS_DIR, exist_ok=True)
os.makedirs(PARQUET_DIR,  exist_ok=True)
os.makedirs(TEMP_DIR,     exist_ok=True)

print(f'담당 시즌  : {MY_SEASONS}')
print(f'X구간 기준 : 최대 {X_MAX_BATTER}타자 이내 투구')
print(f'배치 크기  : {BATCH_SIZE}개')
print(f'zip 저장   : {VIDEO_DIR}')
print(f'체크포인트 : {PROGRESS_CSV}')
print('경로 설정 완료 ✓')

Mounted at /content/drive
담당 시즌  : [2021, 2022, 2023]
X구간 기준 : 최대 9타자 이내 투구
배치 크기  : 200개
zip 저장   : /content/drive/MyDrive/MLB_pitcher/video_zips
체크포인트 : /content/drive/MyDrive/MLB_pitcher/data/progress_2021_2022_2023.csv
경로 설정 완료 ✓


In [3]:
# import gdown, pandas as pd

# gdown.download('https://drive.google.com/uc?id=16OkTmRSl4S8HrVbaGzXQyGagPK5B0tep', '/content/play_ids_2021_2022_2023.csv', quiet=False)
# gdown.download('https://drive.google.com/uc?id=1NknMUmQ8_-oVFfhmnIw0eVQBm1Y-rpM2', '/content/play_ids_2024_2025.csv', quiet=False)

# df1 = pd.read_csv('/content/play_ids_2021_2022_2023.csv')
# df2 = pd.read_csv('/content/play_ids_2024_2025.csv')

# total = pd.concat([df1, df2]).drop_duplicates('play_id').reset_index(drop=True)
# print(f'2021~2023: {len(df1):,}개')
# # print(f'2024~2025: {len(df2):,}개'
# print(f'전체 합계: {len(total):,}개')


In [4]:
# total.to_csv('/content/play_ids_all.csv', index=False)

# # 드라이브에도 저장 (파일 ID 방식 쓰는 공유드라이브면 로컬에만 저장하고 수동 업로드)
# import shutil
# shutil.copy('/content/play_ids_all.csv', '/content/drive/MyDrive/MLB_pitcher/data/play_ids_all.csv')
# print('저장 완료')

In [5]:
!pip install -q requests beautifulsoup4 gdown
import requests
from bs4 import BeautifulSoup
import pandas as pd
import gdown
import re
import time
import zipfile
import shutil
from pathlib import Path

## 1. Statcast Search → play_id 수집

In [6]:
# REQ_HEADERS = {
#     'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
#                   'AppleWebKit/537.36 (KHTML, like Gecko) '
#                   'Chrome/120.0.0.0 Safari/537.36'
# }

# PITCHER_CRAWLED_CSV = os.path.join(PROGRESS_DIR, f'pitcher_crawled_{SEASON_KEY}.csv')

# def build_statcast_url(pitcher_id, season):
#     # 1~3이닝 필터로 X구간(9타자 이내) 근사
#     return (
#         f'https://baseballsavant.mlb.com/statcast_search'
#         f'?hfPT=&hfAB=&hfGT=R%7C&hfPR=&hfZ=&hfStadium=&hfBBL=&hfNewZones='
#         f'&hfPull=&hfC=&hfSea={season}%7C&hfSit=&player_type=pitcher'
#         f'&hfOuts=&home_road=&pitcher_throws=&batter_stands=&hfSA='
#         f'&hfEventOuts=&hfEventRuns=&game_date_gt=&game_date_lt=&hfMo='
#         f'&hfTeam=&hfOpponent=&hfRO=&position=&hfInfield=&hfOutfield='
#         f'&hfInn=1%7C2%7C3%7C&hfBBT=&hfFlag='
#         f'&pitchers_lookup%5B%5D={pitcher_id}'
#         f'&metric_1=&group_by=name&min_pitches=0&min_results=0&min_pas=0'
#         f'&sort_col=pitches&player_event_sort=api_p_release_speed&sort_order=desc'
#         f'&type=details&player_id={pitcher_id}'
#     )

# def crawl_play_ids_for_pitcher(pitcher_id, season):
#     url = build_statcast_url(pitcher_id, season)
#     try:
#         res = requests.get(url, headers=REQ_HEADERS, timeout=30)
#         if res.status_code != 200:
#             return []
#         soup = BeautifulSoup(res.text, 'html.parser')
#         play_ids = []
#         for tag in soup.find_all('a', href=True):
#             m = re.search(r'playId=([0-9a-f\-]{36})', tag['href'])
#             if m:
#                 play_ids.append(m.group(1))
#         return list(set(play_ids))
#     except Exception:
#         return []

# # ── play_id CSV가 이미 있으면 건너뜀 ─────────────────────────────────────────
# if os.path.exists(PLAY_ID_CSV):
#     play_df = pd.read_csv(PLAY_ID_CSV)
#     print(f'기존 play_id 로드: {len(play_df):,}개 → 크롤링 건너뜀')
# else:
#     # parquet 다운로드
#     for season in MY_SEASONS:
#         parquet_path = os.path.join(PARQUET_DIR, f'statcast_{season}.parquet')
#         if os.path.exists(parquet_path):
#             print(f'[{season}] parquet 이미 존재')
#             continue
#         file_id = PARQUET_FILE_IDS[season]
#         print(f'[{season}] parquet 다운로드 중...')
#         gdown.download(f'https://drive.google.com/uc?id={file_id}', parquet_path, quiet=False)
#         print(f'[{season}] 완료')

#     # 선발 pitcher_id 목록 추출 (at_bat_number 최솟값 == 1)
#     pitcher_season_pairs = []
#     for season in MY_SEASONS:
#         parquet_path = os.path.join(PARQUET_DIR, f'statcast_{season}.parquet')
#         if not os.path.exists(parquet_path):
#             continue
#         df = pd.read_parquet(parquet_path, columns=['game_pk', 'pitcher', 'at_bat_number', 'game_type'])
#         df = df[df['game_type'] == 'R']
#         ab_min = (
#             df.groupby(['game_pk', 'pitcher'])['at_bat_number']
#             .min().reset_index()
#             .rename(columns={'at_bat_number': 'min_ab'})
#         )
#         starters = ab_min[ab_min['min_ab'] == 1]['pitcher'].unique()
#         for pid in starters:
#             pitcher_season_pairs.append({'pitcher_id': pid, 'season': season})
#         print(f'[{season}] 선발 투수: {len(starters):,}명')

#     pitcher_df = pd.DataFrame(pitcher_season_pairs)
#     print(f'\n전체 크롤링 대상: {len(pitcher_df):,}건 (투수×시즌)')

#     # 체크포인트 로드
#     if os.path.exists(PITCHER_CRAWLED_CSV):
#         done_df = pd.read_csv(PITCHER_CRAWLED_CSV)
#         done_keys    = set(zip(done_df['pitcher_id'], done_df['season']))
#         play_records = done_df.to_dict('records')
#         print(f'체크포인트 로드: {len(done_keys):,}건 완료')
#     else:
#         done_keys    = set()
#         play_records = []

#     remaining_pairs = pitcher_df[
#         ~pitcher_df.apply(lambda r: (r['pitcher_id'], r['season']) in done_keys, axis=1)
#     ].reset_index(drop=True)
#     print(f'남은 크롤링: {len(remaining_pairs):,}건')

#     for i, row in remaining_pairs.iterrows():
#         pid    = row['pitcher_id']
#         season = row['season']
#         ids    = crawl_play_ids_for_pitcher(pid, season)
#         for play_id in ids:
#             play_records.append({'play_id': play_id, 'pitcher_id': pid, 'season': season})
#         done_keys.add((pid, season))

#         if (i + 1) % 50 == 0 or (i + 1) == len(remaining_pairs):
#             pd.DataFrame(play_records).to_csv(PITCHER_CRAWLED_CSV, index=False)
#             print(f'  [{i+1}/{len(remaining_pairs)}] 체크포인트 저장 ({len(play_records):,}개 play_id)')

#         time.sleep(1.0)

#     play_df = pd.DataFrame(play_records).drop_duplicates('play_id').reset_index(drop=True)
#     play_df.to_csv(PLAY_ID_CSV, index=False)
#     print(f'\nplay_id 저장 완료: {len(play_df):,}개 → {PLAY_ID_CSV}')

## 2. 체크포인트 로드 (코랩 담당: 0~25% 구간)

In [7]:
# ══════════════════════════════════════════════════════
# ★ 코랩 A = 0 / 코랩 B = 1 ★
MY_SLOT = 1
# ══════════════════════════════════════════════════════

TOTAL     = 1_185_600
PC_COUNT  = 5
SLOT_SIZE = TOTAL // PC_COUNT          # 237,120
IDX_START = MY_SLOT * SLOT_SIZE
IDX_END   = IDX_START + SLOT_SIZE if MY_SLOT < PC_COUNT - 1 else TOTAL

PLAY_ID_ALL  = '/content/play_ids_all.csv'
PROGRESS_CSV = os.path.join(PROGRESS_DIR, f'progress_slot_{MY_SLOT}.csv')

# play_ids_all.csv 없으면 파일 ID로 다운로드
if not os.path.exists(PLAY_ID_ALL):
    gdown.download('https://drive.google.com/uc?id=1hXM8ixbPg14c5ih4AWpjGheewELRCG8M',
                   PLAY_ID_ALL, quiet=False)

all_play_df = pd.read_csv(PLAY_ID_ALL)
slot_df     = all_play_df.iloc[IDX_START:IDX_END].reset_index(drop=True)

if os.path.exists(PROGRESS_CSV):
    progress = pd.read_csv(PROGRESS_CSV)
    done_ids = set(progress[progress['status'] == 'done']['play_id'].tolist())
    print(f'체크포인트 로드: 완료 {len(done_ids):,}개')
else:
    progress = pd.DataFrame(columns=['play_id', 'season', 'batch_id', 'status'])
    done_ids = set()
    print('체크포인트 없음 → 처음부터 시작')

remaining = slot_df[~slot_df['play_id'].isin(done_ids)].reset_index(drop=True)
print(f'슬롯         : {MY_SLOT} (코랩)')
print(f'담당 구간    : {IDX_START:,} ~ {IDX_END:,}')
print(f'전체 담당    : {len(slot_df):,}개')
print(f'남은 play_id : {len(remaining):,}개')
print(f'예상 소요    : 약 {len(remaining) * 4 / 3600:.1f}시간')

Downloading...
From: https://drive.google.com/uc?id=1hXM8ixbPg14c5ih4AWpjGheewELRCG8M
To: /content/play_ids_all.csv
100%|██████████| 58.1M/58.1M [00:00<00:00, 80.7MB/s]


체크포인트 없음 → 처음부터 시작
슬롯         : 1 (코랩)
담당 구간    : 237,120 ~ 474,240
전체 담당    : 237,120개
남은 play_id : 237,120개
예상 소요    : 약 263.5시간


In [ ]:
REQ_HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/120.0.0.0 Safari/537.36'
}

def get_cdn_url(play_id):
    url  = f'https://baseballsavant.mlb.com/sporty-videos?playId={play_id}'
    res  = requests.get(url, headers=REQ_HEADERS, timeout=15)
    soup = BeautifulSoup(res.text, 'html.parser')
    video = soup.find('video')
    if video and video.find('source'):
        return video.find('source').get('src')
    return None

def download_mp4(cdn_url, save_path):
    with requests.get(cdn_url, headers=REQ_HEADERS, stream=True, timeout=30) as r:
        r.raise_for_status()
        with open(save_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)

# 슬롯별 zip 구분 (다른 슬롯과 안 겹치게)
existing_zips = list(Path(VIDEO_DIR).glob(f'batch_slot{MY_SLOT}_*.zip'))
start_batch   = len(existing_zips)
new_records   = []

for batch_idx, batch_start in enumerate(range(0, len(remaining), BATCH_SIZE)):
    batch_num  = start_batch + batch_idx + 1
    batch_name = f'batch_slot{MY_SLOT}_{batch_num:04d}'
    zip_path   = os.path.join(VIDEO_DIR, f'{batch_name}.zip')

    if os.path.exists(zip_path):
        print(f'[{batch_name}] 이미 존재 → 건너뜀')
        continue

    batch_df  = remaining.iloc[batch_start:batch_start + BATCH_SIZE]
    batch_dir = os.path.join(TEMP_DIR, batch_name)
    os.makedirs(batch_dir, exist_ok=True)

    print(f'\n[{batch_name}] {len(batch_df)}개 다운로드 시작...')
    batch_done = 0
    batch_fail = 0

    for i, (_, row) in enumerate(batch_df.iterrows()):
        play_id  = row['play_id']
        season   = row['season']
        mp4_path = os.path.join(batch_dir, f'{play_id}.mp4')

        try:
            cdn_url = get_cdn_url(play_id)
            if cdn_url:
                download_mp4(cdn_url, mp4_path)
                new_records.append({'play_id': play_id, 'season': season,
                                    'batch_id': batch_name, 'status': 'done'})
                batch_done += 1
            else:
                new_records.append({'play_id': play_id, 'season': season,
                                    'batch_id': batch_name, 'status': 'fail'})
                batch_fail += 1
        except Exception as e:
            new_records.append({'play_id': play_id, 'season': season,
                                'batch_id': batch_name, 'status': 'fail'})
            batch_fail += 1

        if (i + 1) % 10 == 0:
            print(f'  {i+1}/{len(batch_df)} | 성공 {batch_done} / 실패 {batch_fail}')

        time.sleep(1.0)

    # zip 묶기
    print(f'[{batch_name}] zip 압축 중...')
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for mp4 in Path(batch_dir).glob('*.mp4'):
            zf.write(mp4, mp4.name)

    zip_mb = os.path.getsize(zip_path) / 1024 / 1024
    print(f'[{batch_name}] 완료 ({zip_mb:.1f} MB) 성공 {batch_done} / 실패 {batch_fail}')

    shutil.rmtree(batch_dir)

    # 체크포인트 저장
    progress = pd.concat([progress, pd.DataFrame(new_records)], ignore_index=True)
    progress.to_csv(PROGRESS_CSV, index=False)
    new_records = []

print('\n전체 다운로드 완료!')


[batch_slot1_0001] 200개 다운로드 시작...
  10/200 | 성공 10 / 실패 0
  20/200 | 성공 20 / 실패 0
  30/200 | 성공 30 / 실패 0
  40/200 | 성공 40 / 실패 0
  50/200 | 성공 50 / 실패 0
  60/200 | 성공 60 / 실패 0
  70/200 | 성공 70 / 실패 0
  80/200 | 성공 80 / 실패 0
  90/200 | 성공 90 / 실패 0
  100/200 | 성공 100 / 실패 0
  110/200 | 성공 110 / 실패 0
  120/200 | 성공 120 / 실패 0
  130/200 | 성공 130 / 실패 0
  140/200 | 성공 140 / 실패 0
  150/200 | 성공 149 / 실패 1
  160/200 | 성공 159 / 실패 1
  170/200 | 성공 169 / 실패 1
  180/200 | 성공 179 / 실패 1
  190/200 | 성공 189 / 실패 1
  200/200 | 성공 199 / 실패 1
[batch_slot1_0001] zip 압축 중...
[batch_slot1_0001] 완료 (1006.4 MB) 성공 199 / 실패 1

[batch_slot1_0002] 200개 다운로드 시작...
  10/200 | 성공 10 / 실패 0
  20/200 | 성공 20 / 실패 0
  30/200 | 성공 30 / 실패 0
  40/200 | 성공 40 / 실패 0
  50/200 | 성공 50 / 실패 0
  60/200 | 성공 60 / 실패 0
  70/200 | 성공 70 / 실패 0
  80/200 | 성공 80 / 실패 0
  90/200 | 성공 90 / 실패 0
  100/200 | 성공 100 / 실패 0
  110/200 | 성공 110 / 실패 0
  120/200 | 성공 120 / 실패 0
  130/200 | 성공 130 / 실패 0
  140/200 | 성공 140 / 실패 0
 

KeyboardInterrupt: 

## 4. 현황 확인

In [ ]:
progress = pd.read_csv(PROGRESS_CSV)
print(f'=== 슬롯 {MY_SLOT} 다운로드 현황 ===')
print(progress['status'].value_counts())
print(f'\n시즌별 현황:')
print(progress.groupby(['season', 'status']).size().unstack(fill_value=0))

zips = sorted(Path(VIDEO_DIR).glob(f'batch_slot{MY_SLOT}_*.zip'))
total_gb = sum(z.stat().st_size for z in zips) / 1024**3
print(f'\n저장된 zip: {len(zips)}개 / 총 {total_gb:.2f} GB')

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/MLB_pitcher/data/progress_2021_2022_2023.csv'